# Insurance Fraud Detection - Google Colab Deployment
## Run this notebook directly on Google Colab (Free GPU included!)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahidarmaan594/insurance-fraud-detection-using-ml/blob/main/colab_deployment.ipynb)

## Step 1: Clone Repository

In [ ]:
!git clone https://github.com/Shahidarmaan594/insurance-fraud-detection-using-ml.git
%cd insurance-fraud-detection-using-ml
!ls -la

## Step 2: Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
print("✅ All dependencies installed!")

## Step 3: Load Pre-trained Model

In [ ]:
import pickle
import pandas as pd
import numpy as np
from pathlib import Path

# Load the trained model
model_path = 'models/fraud_model.pkl'
with open(model_path, 'rb') as f:
    model = pickle.load(f)

print(f"✅ Model loaded successfully!")
print(f"Model type: {type(model)}")

## Step 4: Test Model with Sample Data

In [ ]:
# Create sample test data
sample_data = pd.DataFrame({
    'age': [35],
    'policy_years': [5],
    'claim_amount': [5000],
    'claim_frequency': [2],
    # Add more features as per your model
})

# Make prediction
prediction = model.predict(sample_data)
probability = model.predict_proba(sample_data)

print(f"🔍 Prediction: {'FRAUD' if prediction[0] == 1 else 'LEGITIMATE'}")
print(f"📊 Confidence: {probability[0][1]:.2%}")

## Step 5: Interactive Prediction Interface

In [ ]:
from flask import Flask, jsonify, request
from flask_cors import CORS
import json

# Create Flask app for Colab
app = Flask(__name__)
CORS(app)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        df = pd.DataFrame([data])
        
        prediction = model.predict(df)
        probability = model.predict_proba(df)
        
        return jsonify({
            'fraud': int(prediction[0]),
            'confidence': float(probability[0][1]),
            'label': 'FRAUD' if prediction[0] == 1 else 'LEGITIMATE'
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy', 'model': 'ready'})

print("✅ Flask API configured!")

## Step 6: Run Interactive Web UI on Colab

In [ ]:
# Install and setup ngrok for public URL
!pip install -q pyngrok

from pyngrok import ngrok

# Get ngrok auth token from https://dashboard.ngrok.com/auth/your-authtoken
# For demo, you can use without auth (limited time)

# Start Flask app in background
from threading import Thread
import time

def run_app():
    app.run(port=5000, debug=False)

thread = Thread(target=run_app, daemon=True)
thread.start()
time.sleep(3)

# Create public URL
public_url = ngrok.connect(5000)
print(f"✅ Your app is now accessible at: {public_url}")
print(f"\n🌐 API Endpoint: {public_url}/predict")
print(f"💚 Health Check: {public_url}/health")

## Step 7: Test API Endpoint

In [ ]:
import requests

# Test the API
test_data = {
    'age': 35,
    'policy_years': 5,
    'claim_amount': 5000,
    'claim_frequency': 2
}

response = requests.post(f'{public_url}/predict', json=test_data)
print("API Response:")
print(json.dumps(response.json(), indent=2))

## Step 8: Keep App Running

In [ ]:
# Keep the cell running (Ctrl+Shift+L to reconnect if disconnected)
import time
print("🚀 App is running! Your public URL:")
print(f"\n{public_url}\n")
print("⏰ Colab will keep this running as long as this cell is active.")
print("❌ Close/Stop cell to shutdown the app.")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\n🛑 App stopped")